<a href="https://colab.research.google.com/github/noelmtv/Portfolio-Tools/blob/main/Fundamental_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =============================================================================
# FUNDAMENTAL ANALYSIS DASHBOARD — Google Colab Notebook
# =============================================================================
# Data source: Financial Modeling Prep (FMP) API — /stable/ endpoints
# Author: Noel Mutevhe
# =============================================================================

# Setup and API Key

In [2]:
!pip install plotly pandas requests --quiet

import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML, Markdown
import warnings
warnings.filterwarnings('ignore')

# Ask for API key securely
from google.colab import userdata
FMP_API_KEY = userdata.get('FMP')
BASE_URL = "https://financialmodelingprep.com/stable"

def fmp_get(endpoint: str, params: dict = None) -> list | dict:
    """
    Generic FMP API caller with error handling for premium restrictions.
    Returns parsed JSON or an empty list/dict on failure.
    """
    url = f"{BASE_URL}/{endpoint}"
    all_params = {"apikey": FMP_API_KEY}
    if params:
        all_params.update(params)
    try:
        resp = requests.get(url, params=all_params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        # FMP sometimes returns a dict with "Error Message"
        if isinstance(data, dict) and "Error Message" in data:
            print(f"⚠️  {endpoint}: {data['Error Message']}")
            return [] if 'statement' in endpoint or 'segmentation' in endpoint or 'earnings' in endpoint or 'ratios' in endpoint or 'metrics' in endpoint else {}
        return data
    except requests.exceptions.HTTPError as e:
        if resp.status_code == 402:
            print(f"🔒 {endpoint}: Requires a Premium FMP subscription. Skipping.")
        else:
            print(f"❌ {endpoint}: HTTP {resp.status_code} — {e}")
        return [] if 'statement' in endpoint or 'segmentation' in endpoint or 'earnings' in endpoint or 'ratios' in endpoint or 'metrics' in endpoint else {}
    except Exception as e:
        print(f"❌ {endpoint}: {e}")
        return []

def safe_df(data, columns=None):
    """Convert API response to DataFrame, optionally selecting columns."""
    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data)
    if columns:
        existing = [c for c in columns if c in df.columns]
        df = df[existing]
    return df

def fmt_large_number(x):
    """Format large numbers for tables."""
    if pd.isna(x):
        return "—"
    if isinstance(x, (int, float)):
        if abs(x) >= 1e12:
            return f"${x/1e12:.2f}T"
        if abs(x) >= 1e9:
            return f"${x/1e9:.2f}B"
        if abs(x) >= 1e6:
            return f"${x/1e6:.2f}M"
        if abs(x) >= 1e3:
            return f"${x/1e3:.2f}K"
        return f"${x:,.2f}"
    return str(x)

print("✅ Setup complete. API key loaded securely.")

# Input Ticker
TICKER = input("Enter stock ticker (e.g., AAPL): ").strip().upper()
if not TICKER:
    TICKER = "AAPL"
    print(f"No ticker entered. Defaulting to {TICKER}.")


✅ Setup complete. API key loaded securely.
Enter stock ticker (e.g., AAPL): BA


# Company Profile

In [3]:
display(Markdown(f"## 📋 Company Profile — {TICKER}"))

profile_data = fmp_get("profile", {"symbol": TICKER})
if profile_data and isinstance(profile_data, list):
    p = profile_data[0]
    profile_df = pd.DataFrame([{
        "Field": k,
        "Value": v
    } for k, v in p.items() if k not in ["description", "image"]])
    # Style the profile table
    display(profile_df.style.set_properties(**{'text-align': 'left'})
            .hide(axis='index')
            .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}]))
    if p.get("description"):
        display(Markdown(f"**Business Description:** {p['description']}"))
else:
    print("⚠️  Could not retrieve company profile.")

## 📋 Company Profile — BA

Field,Value
symbol,BA
price,184.450000
marketCap,145783744088
beta,1.257000
lastDividend,0
range,176.77-254.35
change,-3.100000
changePercentage,-1.652890
volume,1754600
averageVolume,6369910


**Business Description:** The Boeing Company is a global aerospace powerhouse specializing in the design, development, manufacture, sale, and comprehensive support of commercial airliners, military aircraft, satellites, missile defense systems, human space flight, and launch technologies, along with related services across the globe. Its operations are organized into four key segments. The Commercial Airplanes division delivers commercial jet aircraft for passenger and cargo transport, alongside essential fleet support services. The Defense, Space & Security segment concentrates on the research, development, production, and modification of manned and unmanned military aircraft, advanced weapons systems, strategic defense and intelligence solutions (including missile defense, command, control, communications, computers, intelligence, surveillance, and reconnaissance, cyber, and information solutions), and satellite systems for both governmental and commercial use, encompassing space exploration. The Global Services segment provides a vast array of support, such as supply chain and logistics management, engineering, maintenance, upgrades, spare parts, pilot and maintenance training, technical documentation, and data analytics for its commercial and defense clientele. Lastly, the Boeing Capital segment offers financing services, overseeing a portfolio of equipment under various lease and financing structures. Founded in 1916, the company is headquartered in Chicago, Illinois.

# Quick Summary

In [4]:
display(Markdown("## 🧾 Quick Summary"))

if profile_data and isinstance(profile_data, list):
    p = profile_data[0]
    summary = pd.DataFrame({
        "Metric": ["Company", "Sector", "Industry", "Market Cap", "Price", "Beta"],
        "Value": [
            p.get("companyName", "—"),
            p.get("sector", "—"),
            p.get("industry", "—"),
            fmt_large_number(p.get("marketCap")),
            f"${p.get('price', '—')}" if p.get("price") else "—",
            p.get("beta", "—")
        ]
    })
    display(summary.style.hide(axis='index'))

print("\n✅ Fundamental analysis dashboard complete.")
print("💡 Tip: If some sections are empty, they may require a Premium FMP subscription.")

## 🧾 Quick Summary

Metric,Value
Company,The Boeing Company
Sector,Industrials
Industry,Aerospace & Defense
Market Cap,$145.78B
Price,$184.45
Beta,1.257000



✅ Fundamental analysis dashboard complete.
💡 Tip: If some sections are empty, they may require a Premium FMP subscription.


# Earnings History

In [5]:
display(Markdown(f"## 📈 Earnings History — {TICKER}"))

earnings_raw = fmp_get("earnings", {"symbol": TICKER, "limit": 40})
earnings_df = safe_df(earnings_raw)

if not earnings_df.empty:
    # Clean and sort
    earnings_df["date"] = pd.to_datetime(earnings_df["date"], errors="coerce")
    earnings_df = earnings_df.sort_values("date").reset_index(drop=True)

    # Determine available EPS columns
    eps_cols = [c for c in earnings_df.columns if "eps" in c.lower()]
    rev_cols = [c for c in earnings_df.columns if "revenue" in c.lower()]

    # Interactive earnings surprise chart (if both actual and estimated exist)
    actual_col = next((c for c in eps_cols if "actual" in c.lower()), None)
    est_col = next((c for c in eps_cols if "estimated" in c.lower()), None)

    if actual_col and est_col:
        fig_eps = go.Figure()
        fig_eps.add_trace(go.Bar(
            x=earnings_df["date"], y=earnings_df[actual_col],
            name="EPS Actual", marker_color="#2E86AB"
        ))
        fig_eps.add_trace(go.Scatter(
            x=earnings_df["date"], y=earnings_df[est_col],
            mode="lines+markers", name="EPS Estimate",
            line=dict(color="#F18F01", dash="dash")
        ))
        fig_eps.update_layout(
            title=f"{TICKER} — EPS Actual vs. Estimate",
            xaxis_title="Date", yaxis_title="EPS ($)",
            hovermode="x unified", template="plotly_white",
            legend=dict(orientation="h", yanchor="bottom", y=1.02)
        )
        fig_eps.show()
    else:
        # Fallback: plot whatever EPS column is available
        plot_col = eps_cols[0] if eps_cols else None
        if plot_col:
            fig = px.bar(earnings_df, x="date", y=plot_col,
                         title=f"{TICKER} — {plot_col}",
                         template="plotly_white")
            fig.show()

    # Display clean earnings table
    show_cols = ["date"] + [c for c in eps_cols + rev_cols if c in earnings_df.columns]
    display(earnings_df[show_cols].tail(12).style.format(precision=2))
else:
    print("⚠️  No earnings data available (may require Premium).")

## 📈 Earnings History — BA

🔒 earnings: Requires a Premium FMP subscription. Skipping.
⚠️  No earnings data available (may require Premium).


#Revenue Breakdown

In [7]:
# =============================================================================
# SECTION 5: PRODUCT AND GEOGRAPHIC REVENUE SEGMENTATION
# =============================================================================

import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display, Markdown


display(Markdown(f"# Revenue Segmentation — {TICKER}"))


# =============================================================================
# 5.1 NORMALIZE FMP REVENUE SEGMENTATION DATA
# =============================================================================

def normalize_fmp_segmentation(raw_data):
    """
    Convert FMP revenue segmentation data into a clean DataFrame.

    FMP's stable segmentation endpoints can return records in this format:

    [
        {
            "symbol": "AAPL",
            "fiscalYear": 2025,
            "period": "FY",
            "reportedCurrency": "USD",
            "date": "2025-09-27",
            "data": {
                "iPhone": 209586000000,
                "Mac": 33708000000,
                "Services": 109158000000
            }
        }
    ]

    This function extracts the nested revenue values and combines them
    with the corresponding reporting dates and metadata.

    Parameters
    ----------
    raw_data : list, dict, or DataFrame
        Raw response from fmp_get().

    Returns
    -------
    pandas.DataFrame
        One row per reporting period, with one column per segment.
    """

    if raw_data is None:
        return pd.DataFrame()

    # Convert a DataFrame back into records if necessary.
    if isinstance(raw_data, pd.DataFrame):
        records = raw_data.to_dict(orient="records")

    elif isinstance(raw_data, dict):

        # Handle APIs or wrappers that return the records inside a key.
        if isinstance(raw_data.get("data"), list):
            records = raw_data["data"]

        elif isinstance(raw_data.get("results"), list):
            records = raw_data["results"]

        else:
            records = [raw_data]

    elif isinstance(raw_data, list):
        records = raw_data

    else:
        print(f"Unexpected response type: {type(raw_data)}")
        return pd.DataFrame()

    if not records:
        return pd.DataFrame()

    normalized_records = []

    for record in records:

        # Skip unexpected record types.
        if not isinstance(record, dict):
            continue

        # Extract metadata separately from revenue segments.
        normalized_record = {
            "date": record.get("date"),
            "fiscalYear": record.get("fiscalYear"),
            "period": record.get("period"),
            "symbol": record.get("symbol"),
            "reportedCurrency": record.get("reportedCurrency")
        }

        # FMP normally stores individual revenue segments in "data".
        nested_data = record.get("data")

        if isinstance(nested_data, dict):

            # Add the individual products or geographic regions.
            for segment_name, revenue in nested_data.items():
                normalized_record[segment_name] = revenue

        else:
            # Fallback for responses already in flat format.
            metadata_columns = {
                "date",
                "fiscalYear",
                "period",
                "symbol",
                "reportedCurrency",
                "data"
            }

            for column, value in record.items():
                if column not in metadata_columns:
                    normalized_record[column] = value

        normalized_records.append(normalized_record)

    df = pd.DataFrame(normalized_records)

    if df.empty:
        return df

    # Parse reporting dates.
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # Sort by date, with older reporting periods first.
    if "date" in df.columns:
        df = df.sort_values("date", na_position="last")

    # Remove duplicate reporting dates, if any.
    # Keep the final occurrence rather than counting a period twice.
    if "date" in df.columns and df["date"].notna().any():
        df = df.drop_duplicates(subset=["date"], keep="last")

    return df.reset_index(drop=True)


# =============================================================================
# 5.2 PREPARE THE DATA FOR CHARTING
# =============================================================================

def prepare_segmentation_for_chart(df):
    """
    Identify numeric revenue columns and convert the DataFrame to long format.

    Metadata such as symbol, fiscalYear, period and reportedCurrency
    must never be treated as revenue segments.
    """

    if df.empty:
        return pd.DataFrame(), []

    # Explicit list of columns that describe the reporting record,
    # rather than individual revenue segments.
    metadata_columns = {
        "date",
        "fiscalYear",
        "period",
        "symbol",
        "reportedCurrency",
        "currency",
        "calendarYear",
        "acceptedDate",
        "fillingDate",
        "link",
        "finalLink",
        "data"
    }

    # Select candidate segment columns.
    candidate_columns = [
        column
        for column in df.columns
        if column not in metadata_columns
    ]

    # Convert candidate columns to numeric values.
    # This also handles revenue values that arrive as strings.
    for column in candidate_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    # Keep columns containing at least one valid numeric observation.
    segment_columns = [
        column
        for column in candidate_columns
        if df[column].notna().any()
    ]

    # Reject negative values for a conventional revenue composition chart.
    # Negative reported values can arise in unusual restatements or
    # accounting adjustments, so we flag them rather than silently plot them.
    negative_columns = [
        column
        for column in segment_columns
        if (df[column].dropna() < 0).any()
    ]

    if negative_columns:
        print(
            "Warning: Negative values found in these segments:",
            negative_columns
        )

    if not segment_columns:
        return pd.DataFrame(), []

    # Use the actual reporting date whenever available.
    if "date" in df.columns:
        date_column = "date"
    elif "fiscalYear" in df.columns:
        date_column = "fiscalYear"
    else:
        print("No reporting date or fiscal year was found.")
        return pd.DataFrame(), []

    # Melt the wide DataFrame into long format.
    # Each row will represent one segment in one reporting period.
    long_df = df.melt(
        id_vars=[date_column],
        value_vars=segment_columns,
        var_name="Segment",
        value_name="Revenue"
    )

    long_df = long_df.dropna(subset=["Revenue"])

    # Remove records with invalid reporting dates.
    if date_column == "date":
        long_df = long_df.dropna(subset=["date"])

    return long_df, segment_columns


# =============================================================================
# 5.3 CREATE REVENUE CHARTS
# =============================================================================

def plot_revenue_segmentation(
    ticker,
    raw_data,
    segmentation_type,
    currency_fallback="USD"
):
    """
    Normalize, validate and plot product or geographic revenue segmentation.

    Produces:
        1. Stacked revenue chart.
        2. Revenue composition chart (percentage).
        3. A clean underlying revenue table.
    """

    # Normalize the original FMP response.
    df = normalize_fmp_segmentation(raw_data)

    if df.empty:
        print(
            f"No segmentation records were returned for "
            f"{ticker} ({segmentation_type})."
        )
        return

    # Identify revenue columns and prepare the long-format data.
    long_df, segment_columns = prepare_segmentation_for_chart(df)

    if long_df.empty:
        print(
            f"No usable revenue segment columns were found for "
            f"{ticker} ({segmentation_type})."
        )

        print("Available columns:", df.columns.tolist())
        display(df.head())

        return

    # Identify the reporting currency where possible.
    if (
        "reportedCurrency" in df.columns
        and df["reportedCurrency"].notna().any()
    ):
        currencies = df["reportedCurrency"].dropna().unique()

        if len(currencies) == 1:
            currency = currencies[0]
        else:
            currency = "reported currency varies"

    else:
        currency = currency_fallback

    # Select a suitable x-axis.
    x_column = "date" if "date" in long_df.columns else "fiscalYear"

    # -------------------------------------------------------------------------
    # CHART A: STACKED REVENUE
    # -------------------------------------------------------------------------

    fig = px.bar(
        long_df,
        x=x_column,
        y="Revenue",
        color="Segment",
        title=f"{ticker} — Revenue by {segmentation_type}",
        template="plotly_white",
        barmode="stack",
        labels={
            x_column: "Reporting Period",
            "Revenue": f"Revenue ({currency})",
            "Segment": segmentation_type
        },
        hover_data={
            "Revenue": ":,.0f"
        }
    )

    # Improve chart readability.
    fig.update_layout(
        height=600,
        xaxis_title="Financial Reporting Period",
        yaxis_title=f"Revenue ({currency})",
        legend_title=segmentation_type,
        hovermode="x unified",
        margin=dict(l=60, r=30, t=90, b=60)
    )

    fig.update_yaxes(tickformat="~s")

    fig.show()

    # -------------------------------------------------------------------------
    # CHART B: REVENUE COMPOSITION
    # -------------------------------------------------------------------------

    # Calculate the total reported revenue across the segments for each period.
    # This is the sum of reported segments, not necessarily total company
    # revenue. Some companies report unallocated or corporate revenue.
    composition_df = long_df.copy()

    period_totals = (
        composition_df
        .groupby(x_column)["Revenue"]
        .transform("sum")
    )

    # Avoid dividing by zero.
    composition_df["Revenue Share (%)"] = np.where(
        period_totals != 0,
        composition_df["Revenue"] / period_totals * 100,
        np.nan
    )

    fig_pct = px.bar(
        composition_df,
        x=x_column,
        y="Revenue Share (%)",
        color="Segment",
        title=f"{ticker} — {segmentation_type} Revenue Composition",
        template="plotly_white",
        barmode="stack",
        labels={
            x_column: "Reporting Period",
            "Revenue Share (%)": "Share of Reported Segments (%)",
            "Segment": segmentation_type
        }
    )

    fig_pct.update_layout(
        height=550,
        yaxis_title="Share of Reported Segment Revenue (%)",
        legend_title=segmentation_type,
        hovermode="x unified"
    )

    fig_pct.update_yaxes(ticksuffix="%")

    fig_pct.show()

    # -------------------------------------------------------------------------
    # TABLE: REVENUE BY REPORTING PERIOD
    # -------------------------------------------------------------------------

    display(
        Markdown(
            f"### {ticker} — Revenue by {segmentation_type} "
            f"({currency})"
        )
    )

    # Display only the reporting dates and revenue columns.
    display_columns = []

    for column in ["date", "fiscalYear", "period"]:
        if column in df.columns:
            display_columns.append(column)

    display_columns += segment_columns

    display(
        df[display_columns]
        .style
        .format(
            {
                column: "{:,.0f}"
                for column in segment_columns
            },
            na_rep="—"
        )
        .hide(axis="index")
    )

    # -------------------------------------------------------------------------
    # DIAGNOSTICS
    # -------------------------------------------------------------------------

    print(f"Ticker: {ticker}")
    print(f"Segmentation type: {segmentation_type}")
    print(f"Reporting currency: {currency}")
    print(f"Number of reporting periods: {len(df)}")
    print(f"Number of segments: {len(segment_columns)}")
    print(f"Segments identified: {segment_columns}")


# =============================================================================
# 5.4 PRODUCT REVENUE SEGMENTATION
# =============================================================================

display(Markdown("## Product Revenue Segmentation"))

product_raw = fmp_get(
    "revenue-product-segmentation",
    {
        "symbol": TICKER,
        "period": "annual"
    }
)

plot_revenue_segmentation(
    ticker=TICKER,
    raw_data=product_raw,
    segmentation_type="Product",
    currency_fallback="USD"
)


# =============================================================================
# 5.5 GEOGRAPHIC REVENUE SEGMENTATION
# =============================================================================

display(Markdown("## Geographic Revenue Segmentation"))

geo_raw = fmp_get(
    "revenue-geographic-segmentation",
    {
        "symbol": TICKER,
        "period": "annual"
    }
)

plot_revenue_segmentation(
    ticker=TICKER,
    raw_data=geo_raw,
    segmentation_type="Geographic Region",
    currency_fallback="USD"
)

# Revenue Segmentation — BA

## Product Revenue Segmentation

### BA — Revenue by Product (USD)

date,fiscalYear,period,Commercial Airplanes,"Defense, Space & Security",Global Services,Commercial Airplanes Segment,Boeing Defense Space Security Segment,Boeing Capital Corporation,Other Segment
2011-12-31 00:00:00,2011,FY,—,—,—,"36,171,000,000","46,923,000,000","532,000,000","138,000,000"
2012-12-31 00:00:00,2012,FY,—,—,—,"49,127,000,000","32,607,000,000","441,000,000","133,000,000"
2013-12-31 00:00:00,2013,FY,—,—,—,"52,981,000,000","33,197,000,000","408,000,000","102,000,000"
2014-12-31 00:00:00,2014,FY,—,—,—,"59,990,000,000","30,881,000,000","416,000,000",—
2015-12-31 00:00:00,2015,FY,—,—,—,"66,048,000,000","30,388,000,000","413,000,000",—
2016-12-31 00:00:00,2016,FY,—,—,—,"65,069,000,000","29,498,000,000","298,000,000",—
2017-12-31 00:00:00,2017,FY,—,—,"14,639,000,000","56,729,000,000","21,057,000,000","307,000,000",—
2018-12-31 00:00:00,2018,FY,—,—,"17,018,000,000","60,715,000,000","23,195,000,000","274,000,000",—
2019-12-31 00:00:00,2019,FY,—,—,"18,468,000,000","32,255,000,000","26,227,000,000","244,000,000",—
2020-12-31 00:00:00,2020,FY,—,—,"15,543,000,000","16,162,000,000","26,257,000,000","261,000,000",—


Ticker: BA
Segmentation type: Product
Reporting currency: USD
Number of reporting periods: 15
Number of segments: 7
Segments identified: ['Commercial Airplanes', 'Defense, Space & Security', 'Global Services', 'Commercial Airplanes Segment', 'Boeing Defense Space Security Segment', 'Boeing Capital Corporation', 'Other Segment']


## Geographic Revenue Segmentation

### BA — Revenue by Geographic Region (USD)

date,fiscalYear,period,Africa,Asia Other Than China,CANADA,Europe,Latin America,Middle East,Non-US,UNITED STATES,CHINA
2011-12-31 00:00:00,2011,FY,"1,759,000,000","7,438,000,000","618,000,000","9,850,000,000","1,356,000,000","5,477,000,000",—,"34,391,000,000","4,779,000,000"
2012-12-31 00:00:00,2012,FY,"1,282,000,000","10,390,000,000","586,000,000","10,269,000,000","3,555,000,000","10,285,000,000",—,"37,202,000,000","6,086,000,000"
2013-12-31 00:00:00,2013,FY,"621,000,000","12,200,000,000","1,486,000,000","10,622,000,000","2,725,000,000","9,165,000,000",—,"37,592,000,000","10,555,000,000"
2014-12-31 00:00:00,2014,FY,"2,596,000,000","11,900,000,000","1,901,000,000","11,898,000,000","2,596,000,000","9,243,000,000",—,"37,842,000,000","11,029,000,000"
2015-12-31 00:00:00,2015,FY,"1,398,000,000","13,433,000,000","1,870,000,000","12,248,000,000","1,875,000,000","10,846,000,000",—,"39,287,000,000","12,556,000,000"
2016-12-31 00:00:00,2016,FY,"1,999,000,000","10,553,000,000","2,076,000,000","13,790,000,000","1,936,000,000","13,297,000,000",—,"38,765,000,000","10,312,000,000"
2017-12-31 00:00:00,2017,FY,"755,000,000","8,899,000,000","2,197,000,000","11,457,000,000","1,494,000,000","12,287,000,000",—,"42,331,000,000","11,911,000,000"
2018-12-31 00:00:00,2018,FY,"1,486,000,000","12,141,000,000","2,583,000,000","12,976,000,000","1,458,000,000","9,745,000,000",—,"44,676,000,000","13,764,000,000"
2019-12-31 00:00:00,2019,FY,"1,113,000,000","10,662,000,000","2,019,000,000","10,366,000,000","1,015,000,000","9,272,000,000",—,"42,681,000,000","5,684,000,000"
2020-12-31 00:00:00,2020,FY,"114,000,000","4,128,000,000","1,302,000,000","7,961,000,000","229,000,000","5,308,000,000",—,"36,979,000,000","1,803,000,000"


Ticker: BA
Segmentation type: Geographic Region
Reporting currency: USD
Number of reporting periods: 15
Number of segments: 9
Segments identified: ['Africa', 'Asia Other Than China', 'CANADA', 'Europe', 'Latin America', 'Middle East', 'Non-US', 'UNITED STATES', 'CHINA']


# Financial Statements

In [8]:
# =============================================================================
# SECTION: FINANCIAL STATEMENTS
# =============================================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np

display(Markdown(f"# 📊 Financial Statements — {TICKER}"))


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

STATEMENTS = {
    "Income Statement": "income-statement",
    "Balance Sheet": "balance-sheet-statement",
    "Cash Flow Statement": "cash-flow-statement"
}

# Number of reporting periods to display.
NUMBER_OF_PERIODS = 5

# Fields that describe the report rather than individual financial metrics.
METADATA_COLUMNS = {
    "date",
    "symbol",
    "reportedCurrency",
    "cik",
    "fillingDate",
    "filingDate",
    "acceptedDate",
    "calendarYear",
    "period",
    "link",
    "finalLink"
}


# =============================================================================
# 2. FINANCIAL STATEMENT FORMATTING FUNCTION
# =============================================================================

def display_traditional_statement(df, statement_name, periods=5):
    """
    Display an FMP financial statement in a traditional financial-report layout.

    Layout:
        - Rows: financial metrics
        - Columns: reporting periods
        - Latest reporting period first

    Features:
        - Removes unnecessary metadata
        - Formats numbers with thousands separators
        - Displays negative values in parentheses
        - Highlights important subtotals
        - Preserves the financial metrics reported by FMP
    """

    # -------------------------------------------------------------------------
    # STEP 1: Validate the input
    # -------------------------------------------------------------------------

    if df is None or df.empty:
        print(f"⚠️ {statement_name}: No data available.")
        return

    df = df.copy()

    # -------------------------------------------------------------------------
    # STEP 2: Identify the reporting-period column
    # -------------------------------------------------------------------------

    date_column = next(
        (
            col for col in ["date", "calendarYear", "fillingDate", "filingDate"]
            if col in df.columns
        ),
        None
    )

    if date_column is None:
        print(
            f"⚠️ Could not identify the reporting period for "
            f"{statement_name}."
        )
        display(df.head())
        return

    # -------------------------------------------------------------------------
    # STEP 3: Sort the financial statements
    # -------------------------------------------------------------------------

    if date_column in ["fillingDate", "filingDate"]:
        df[date_column] = pd.to_datetime(
            df[date_column],
            errors="coerce"
        )

    elif date_column == "date":
        df[date_column] = pd.to_datetime(
            df[date_column],
            errors="coerce"
        )

    # Latest reporting period first.
    df = df.sort_values(
        date_column,
        ascending=False,
        na_position="last"
    )

    # Keep only the requested number of reporting periods.
    df = df.head(periods).copy()

    # -------------------------------------------------------------------------
    # STEP 4: Generate readable column headings
    # -------------------------------------------------------------------------

    if date_column in ["date", "fillingDate", "filingDate"]:

        report_dates = pd.to_datetime(
            df[date_column],
            errors="coerce"
        )

        period_labels = report_dates.dt.strftime("%Y")

    else:
        period_labels = df[date_column].astype(str)

    # If two reports have the same year, distinguish them.
    period_labels = period_labels.fillna("Unknown").astype(str)

    if period_labels.duplicated().any():
        period_labels = [
            f"{label} ({i + 1})"
            for i, label in enumerate(period_labels)
        ]

    # -------------------------------------------------------------------------
    # STEP 5: Identify financial metrics
    # -------------------------------------------------------------------------

    metric_columns = [
        col for col in df.columns
        if col not in METADATA_COLUMNS
    ]

    # Keep fields containing numeric financial data.
    numeric_columns = []

    for col in metric_columns:

        numeric_values = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        if numeric_values.notna().any():
            numeric_columns.append(col)
            df[col] = numeric_values

    if not numeric_columns:
        print(f"⚠️ No numeric financial metrics found in {statement_name}.")
        return

    # -------------------------------------------------------------------------
    # STEP 6: Transpose the table
    # -------------------------------------------------------------------------

    # Original:
    # One row per year, one column per financial metric.
    #
    # Transposed:
    # One row per financial metric, one column per year.

    statement = df[numeric_columns].T

    statement.columns = period_labels
    statement.index.name = "Financial Metric"

    # -------------------------------------------------------------------------
    # STEP 7: Format financial values
    # -------------------------------------------------------------------------

    def format_financial_value(value):

        if pd.isna(value):
            return "—"

        value = float(value)

        # Use parentheses for negative numbers.
        if value < 0:
            return f"({abs(value):,.0f})"

        return f"{value:,.0f}"

    # -------------------------------------------------------------------------
    # STEP 8: Identify important subtotal rows
    # -------------------------------------------------------------------------

    # These are common FMP field names.
    # Additional labels are included to accommodate different statements.

    subtotal_keywords = [
        "gross profit",
        "operating income",
        "operating profit",
        "income before tax",
        "income before taxes",
        "net income",
        "net profit",
        "ebitda",
        "total revenue",
        "total assets",
        "total liabilities",
        "total stockholders equity",
        "total shareholders equity",
        "total stockholders' equity",
        "total shareholders' equity",
        "total equity",
        "total current assets",
        "total current liabilities",
        "total non current assets",
        "total non current liabilities",
        "net cash provided by operating activities",
        "net cash provided by investing activities",
        "net cash provided by financing activities",
        "free cash flow"
    ]

    # Find the matching row labels.
    subtotal_rows = []

    for row_name in statement.index:

        normalized_name = str(row_name).lower().replace("_", " ").strip()

        if any(keyword in normalized_name for keyword in subtotal_keywords):
            subtotal_rows.append(row_name)

    # -------------------------------------------------------------------------
    # STEP 9: Display the formatted statement
    # -------------------------------------------------------------------------

    display(Markdown(f"### {statement_name}"))

    # Provide the currency context when available.
    if (
        "reportedCurrency" in df.columns
        and df["reportedCurrency"].notna().any()
    ):
        currency = df["reportedCurrency"].dropna().iloc[0]

        display(Markdown(f"**Reporting currency:** {currency}"))

    # Format the data for presentation.
    styled_statement = (
        statement
        .style
        .format(format_financial_value, na_rep="—")
        .set_properties(
            **{
                "text-align": "right",
                "white-space": "nowrap"
            }
        )
        .set_properties(
            subset=pd.IndexSlice[:, :],
            **{"padding": "5px 12px"}
        )
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("text-align", "left"),
                    ("font-weight", "bold"),
                    ("background-color", "#EAF0F6"),
                    ("padding", "7px 12px")
                ]
            },
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "16px"),
                    ("font-weight", "bold")
                ]
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", "100%")
                ]
            }
        ])
        .set_caption(statement_name)
    )

    # Highlight subtotal rows.
    def highlight_subtotals(row):

        if row.name in subtotal_rows:
            return [
                "font-weight: bold; border-top: 1px solid #777;"
                for _ in row
            ]

        return ["" for _ in row]

    styled_statement = styled_statement.apply(
        highlight_subtotals,
        axis=1
    )

    display(styled_statement)


# =============================================================================
# 3. RETRIEVE AND DISPLAY ALL THREE FINANCIAL STATEMENTS
# =============================================================================

for label, endpoint in STATEMENTS.items():

    display(Markdown("---"))

    # Retrieve the annual financial statement from FMP.
    raw = fmp_get(
        endpoint,
        {
            "symbol": TICKER,
            "period": "annual",
            "limit": NUMBER_OF_PERIODS
        }
    )

    # Convert the response into a DataFrame using your existing function.
    df = safe_df(raw)

    if not df.empty:

        display_traditional_statement(
            df=df,
            statement_name=label,
            periods=NUMBER_OF_PERIODS
        )

    else:
        print(
            f"⚠️ {label} unavailable. "
            "Check API access, the response, and your FMP subscription."
        )

# 📊 Financial Statements — BA

---

### Income Statement

**Reporting currency:** USD

date,2025,2024,2023,2022,2021
Financial Metric,,,,,
fiscalYear,"2,025","2,024","2,023","2,022","2,021"
revenue,"89,463,000,000","66,517,000,000","77,794,000,000","66,608,000,000","62,286,000,000"
costOfRevenue,"85,174,000,000","68,508,000,000","70,070,000,000","63,078,000,000","59,237,000,000"
grossProfit,"4,289,000,000","(1,991,000,000)","7,724,000,000","3,530,000,000","3,049,000,000"
researchAndDevelopmentExpenses,"3,615,000,000","3,812,000,000","3,377,000,000","2,852,000,000","2,249,000,000"
generalAndAdministrativeExpenses,"6,090,000,000","5,021,000,000","5,168,000,000","4,187,000,000","4,157,000,000"
sellingAndMarketingExpenses,0,0,0,0,0
sellingGeneralAndAdministrativeExpenses,"6,090,000,000","5,021,000,000","5,168,000,000","4,187,000,000","4,157,000,000"
otherExpenses,"(9,697,000,000)",0,0,0,"(487,000,000)"


...

date,2025,2024,2023,2022,2021
Financial Metric,,,,,
fiscalYear,"2,025","2,024","2,023","2,022","2,021"
cashAndCashEquivalents,"10,921,000,000","13,801,000,000","12,691,000,000","14,614,000,000","8,052,000,000"
shortTermInvestments,"18,479,000,000","12,481,000,000","3,274,000,000","2,606,000,000","8,192,000,000"
cashAndShortTermInvestments,"29,400,000,000","26,282,000,000","15,965,000,000","17,220,000,000","16,244,000,000"
netReceivables,"12,079,000,000","11,201,000,000","11,065,000,000","11,305,000,000","11,378,000,000"
accountsReceivables,"2,697,000,000","2,625,000,000","2,624,000,000","2,512,000,000","2,636,000,000"
otherReceivables,"9,382,000,000","8,576,000,000","8,441,000,000","8,793,000,000","8,742,000,000"
inventory,"84,679,000,000","87,550,000,000","79,741,000,000","78,151,000,000","78,823,000,000"
prepaids,0,0,0,0,0


---

### Cash Flow Statement

**Reporting currency:** USD

date,2025,2024,2023,2022,2021
Financial Metric,,,,,
fiscalYear,"2,025","2,024","2,023","2,022","2,021"
netIncome,"1,890,000,000","(11,829,000,000)","(2,242,000,000)","(4,935,000,000)","(4,164,000,000)"
depreciationAndAmortization,"1,953,000,000","1,836,000,000","1,861,000,000","1,979,000,000","2,144,000,000"
deferredIncomeTax,0,0,0,0,"3,281,000,000"
stockBasedCompensation,"426,000,000","407,000,000","690,000,000","1,941,000,000","2,073,000,000"
changeInWorkingCapital,"(1,002,000,000)","(8,768,000,000)","4,089,000,000","4,139,000,000","(6,977,000,000)"
accountsReceivables,"(95,000,000)","415,000,000","764,000,000","142,000,000","(713,000,000)"
inventory,"(1,501,000,000)","(12,353,000,000)","1,684,000,000","420,000,000","(1,127,000,000)"
accountsPayables,"724,000,000","(793,000,000)","1,672,000,000","838,000,000","(3,783,000,000)"


# EPS History

In [9]:
display(Markdown(f"## 💹 EPS & P/E History — {TICKER}"))

# A. Get historical prices
price_raw = fmp_get("historical-price-eod/full", {
    "symbol": TICKER,
    "from": (pd.Timestamp.now() - pd.DateOffset(years=5)).strftime("%Y-%m-%d"),
    "to": pd.Timestamp.now().strftime("%Y-%m-%d")
})
price_df = safe_df(price_raw)

# B. Get earnings EPS (annual)
eps_raw = fmp_get("earnings", {"symbol": TICKER, "limit": 20})
eps_df = safe_df(eps_raw)

if not price_df.empty and not eps_df.empty:
    price_df["date"] = pd.to_datetime(price_df["date"], errors="coerce")
    eps_df["date"] = pd.to_datetime(eps_df["date"], errors="coerce")

    # Find EPS actual column
    eps_col = next((c for c in eps_df.columns if "eps" in c.lower() and "actual" in c.lower()),
                   next((c for c in eps_df.columns if "eps" in c.lower()), None))

    if eps_col:
        eps_series = eps_df.set_index("date")[eps_col].sort_index()

        # Compute trailing P/E at each price date using the latest reported EPS
        price_df = price_df.sort_values("date")
        price_df["eps_reported"] = np.nan
        for idx, row in price_df.iterrows():
            # Find most recent EPS report on or before this date
            prior_eps = eps_series[eps_series.index <= row["date"]]
            if not prior_eps.empty:
                price_df.at[idx, "eps_reported"] = prior_eps.iloc[-1]

        price_df["pe_ratio"] = price_df["close"] / price_df["eps_reported"]
        price_df = price_df.dropna(subset=["pe_ratio"])

        if not price_df.empty:
            fig_pe = make_subplots(specs=[[{"secondary_y": True}]])
            fig_pe.add_trace(
                go.Scatter(x=price_df["date"], y=price_df["pe_ratio"],
                           name="P/E Ratio", line=dict(color="#2E86AB")),
                secondary_y=False
            )
            # Overlay EPS bars on secondary axis (annual points)
            fig_pe.add_trace(
                go.Bar(x=eps_series.index, y=eps_series.values,
                       name="EPS (Reported)", marker_color="#F18F01", opacity=0.6),
                secondary_y=True
            )
            fig_pe.update_layout(
                title=f"{TICKER} — Historical P/E vs. Reported EPS",
                hovermode="x unified", template="plotly_white",
                legend=dict(orientation="h", yanchor="bottom", y=1.02)
            )
            fig_pe.update_yaxes(title_text="P/E Ratio", secondary_y=False)
            fig_pe.update_yaxes(title_text="EPS ($)", secondary_y=True)
            fig_pe.show()
        else:
            print("⚠️  Could not compute P/E series.")
    else:
        print("⚠️  No EPS column found for P/E calculation.")
else:
    print("⚠️  Price or earnings data insufficient for P/E history.")

## 💹 EPS & P/E History — BA

🔒 earnings: Requires a Premium FMP subscription. Skipping.
⚠️  Price or earnings data insufficient for P/E history.


# Key Metrics and Ratios

In [10]:
display(Markdown(f"## 📐 Key Metrics & Ratios (TTM) — {TICKER}"))

# a. Ratios TTM
ratios_raw = fmp_get("ratios-ttm", {"symbol": TICKER})
ratios_df = safe_df(ratios_raw)

# b. Key Metrics TTM
metrics_raw = fmp_get("key-metrics-ttm", {"symbol": TICKER})
metrics_df = safe_df(metrics_raw)

if not ratios_df.empty:
    # Transpose single-row ratios for readability
    ratios_t = ratios_df.T.reset_index()
    ratios_t.columns = ["Metric", "Value"]
    display(ratios_t.style.format(precision=3).hide(axis='index'))
else:
    print("⚠️  Ratios TTM not available (may require Premium).")

if not metrics_df.empty:
    metrics_t = metrics_df.T.reset_index()
    metrics_t.columns = ["Metric", "Value"]
    display(metrics_t.style.format(precision=3).hide(axis='index'))
else:
    print("⚠️  Key Metrics TTM not available (may require Premium).")

## 📐 Key Metrics & Ratios (TTM) — BA

Metric,Value
symbol,BA
grossProfitMarginTTM,0.047
ebitMarginTTM,0.057
ebitdaMarginTTM,0.080
operatingProfitMarginTTM,-0.054
pretaxProfitMarginTTM,0.030
continuousOperationsProfitMarginTTM,0.027
netProfitMarginTTM,0.026
bottomLineProfitMarginTTM,0.020
receivablesTurnoverTTM,7.134


Metric,Value
symbol,BA
marketCap,145823180579
enterpriseValueTTM,184474180579
evToSalesTTM,1.963
evToOperatingCashFlowTTM,50.694
evToFreeCashFlowTTM,-878.448
evToEBITDATTM,24.447
netDebtToEBITDATTM,5.122
currentRatioTTM,1.138
incomeQualityTTM,1.748
